This week we will be learning about aggregations and using them to do a deeper investigation into the effects of different types of joins. 

We will also do a few problems exploring matrix inverses. 

### Tutorials
- https://www.datacamp.com/community/tutorials/pandas-multi-index
- https://www.datacamp.com/community/tutorials/pandas-split-apply-combine-groupby

## Grouping Data

The expressiveness of Python and pandas allows complex group operations using any function that accepts a pandas object or NumPy array. This can include:

- Splitting a pandas object into pieces using one or more keys
- Calculating group summary statistics
- Applying within-group transformations or other manipulations
- Computing pivot tables and cross-tabulations
- Performing quantile analysis and other statistical group analyses

### GroupBy Operations

Group operations involve the `split-apply-combine` mechanism.

1. Data are split into groups based on one or more keys
2. A function is applied to each group
3. Results of the function application are combined into a new object

Grouping keys can take many forms, and the keys do not have to be all of the same type.

pandas `groupby` method returns a GroupBy object that can be re-used.

DataFrame columns can be used as the group keys.

Numeric aggregations will exclude `nuisance` (non-numeric) columns from the result

By default `groupby` groups on axis=0, but can group on any of the other axes.

### Iterating over groups

The GroupBy object supports iteration, generating a sequence of 2-tuples containing the group name along with the chunk of data.

Indexing a GroupBy object created from a DataFrame with a column name or array of column names has the effect of column subsetting for aggregation. This means that:
```
df.groupby('key1')['data1']
```
is essentially the same as:
```
df['data1'].groupby(df['key1'])
```

### Reindexing

The output of GroupBys are often multi-indexed. This is typically not the form you want for your analysis. This means that you will want to reset your index or remove multi indexes on your columns. See below for examples of that. 

In [1]:
import pandas as pd
import numpy as np

speeds = pd.DataFrame(
    [
        ("bird", "Falconiformes", 389.0, 0),
        ("bird", "Psittaciformes", 24.0, 0),
        ("mammal", "Carnivora", 80.2, 0),
        ("mammal", "Primates", np.nan, 0),
        ("mammal", "Carnivora", 58, 0),
    ],
    index=["falcon", "parrot", "lion", "monkey", "leopard"],
    columns=("class", "order", "max_speed", 'min_speed'),
)

not_reindexed = speeds.groupby(["class", "order"]).agg(['sum', 'mean'])
not_reindexed

max_speed        min_speed     
                            sum   mean       sum mean
class  order                                         
bird   Falconiformes      389.0  389.0         0  0.0
       Psittaciformes      24.0   24.0         0  0.0
mammal Carnivora          138.2   69.1         0  0.0
       Primates             0.0    NaN         0  0.0

In [2]:
not_reindexed.columns = ["_".join(i) for i in not_reindexed.columns]
not_reindexed

max_speed_sum  max_speed_mean  min_speed_sum  \
class  order                                                          
bird   Falconiformes           389.0           389.0              0   
       Psittaciformes           24.0            24.0              0   
mammal Carnivora               138.2            69.1              0   
       Primates                  0.0             NaN              0   

                       min_speed_mean  
class  order                           
bird   Falconiformes              0.0  
       Psittaciformes             0.0  
mammal Carnivora                  0.0  
       Primates                   0.0

In [ ]:
not_reindexed.reset_index()

,class,order,max_speed_sum,max_speed_mean,min_speed_sum,min_speed_mean
0,bird,Falconiformes,389.0,389.0,0,0.0
1,bird,Psittaciformes,24.0,24.0,0,0.0
2,mammal,Carnivora,138.2,69.1,0,0.0
3,mammal,Primates,0.0,NaN,0,0.0


### All Data Descriptions


* `customers` - A table of customers. The unique id for the customer is `customer_id` meaning only one id per customer. 
* `orders` - A table of orders. The unique id for the order is `order_id` there is a foreign key (`customer_id`) that tells you which customer placed the order.
* `web_visits`: Describes visitors to the website, with `visitor_id` be a unique id per visitor to the website. If the visitor is a logged in customer then the `customer_id` has a value otherwise it is null. 
* `ad_clicks`: Describes clicks on ad on an external website, the `external_user_id` is an unique id per user from the external website. If the user is a known in customer then the `customer_id` has a value otherwise it is null. 

In [4]:
import generate_week_10_data

customers, orders, web_visits, ad_clicks = generate_week_10_data.create_data()

FileNotFoundError: [Errno 2] No such file or directory: '/usr/share/dict/words'

### Problem 1 (10 pts)

1. What is the order amount per `customer_name`, only include customers who have orders
2. What is the order amount per `customer_name`, include customers even if they have no orders

#### Grading: 
5 points per question. 2 points for the correct join and 3 point for the aggragations. 

In [ ]:
# Problem 1.1: Order amount per customer_name, only include customers with orders
result_1_1 = customers.merge(orders, on='customer_id', how='inner').groupby('customer_name')['order_amount'].sum()
print("Problem 1.1 - Only customers with orders:")
print(result_1_1)

# Problem 1.2: Order amount per customer_name, include all customers
result_1_2 = customers.merge(orders, on='customer_id', how='left').groupby('customer_name')['order_amount'].sum()
print("\nProblem 1.2 - All customers (including those without orders):")
print(result_1_2)

### Problem 2 (5 pts)
If you look at the raw results of the join you can visually see that there are duplicates for the columns that started in the `customer` table. This wasn't a concern above because we we doing an aggregation on a column in the `orders` table. Let's think through problems that require a join but are looking to do an aggregation with a column from the `customers` table. 

1. What is the average `customer_age` for customers who have placed orders. For this problem print the accurate average `customer_age` with duplicates remove and also print the inaccure average `customer_age` without duplicates removed. 
2. What is the average `customer_age` for customers who have placed orders versus those who haven't. 

#### Grading: 
5 points per question. 2 points for the correct join and 3 point for the aggragations. 

In [ ]:
# Problem 2.1: Average customer age for customers with orders
merged_2 = customers.merge(orders, on='customer_id', how='inner')

# Inaccurate average (with duplicates)
inaccurate_avg = merged_2['customer_age'].mean()
print("Problem 2.1 - Inaccurate average (with duplicates):", inaccurate_avg)

# Accurate average (duplicates removed)
accurate_avg = merged_2.drop_duplicates(subset=['customer_id'])['customer_age'].mean()
print("Problem 2.1 - Accurate average (duplicates removed):", accurate_avg)

# Problem 2.2: Average age for customers with orders vs without orders
customers_with_orders = customers.merge(orders, on='customer_id', how='inner').drop_duplicates(subset=['customer_id'])
customers_without_orders = customers.merge(orders, on='customer_id', how='left')
customers_without_orders = customers_without_orders[customers_without_orders['order_id'].isna()]

print("\nProblem 2.2:")
print("Average age for customers WITH orders:", customers_with_orders['customer_age'].mean())
print("Average age for customers WITHOUT orders:", customers_without_orders['customer_age'].mean())

### Problem 3 

Per customer what is the total number of web visits and ads clicked on? 

For this problem I want you to start with the `customers_visits_clicks_raw` table below and again do an accurate count that only counts a web visit once and an inaccurate count that counts duplications. Here is what the results should look like: 

<img src="images/number_visits_clicks.png" width="800"/>


#### Grading: 
5 points for joining correctly. 5 points for the correct aggragtions.  


In [ ]:
# Start with this dataset 
customers_visits_clicks_raw = customers.merge(web_visits, on="customer_id", how="left").merge(ad_clicks, on="customer_id", how="left")

# Inaccurate counts (with duplicates)
inaccurate = customers_visits_clicks_raw.groupby('customer_name').agg({
    'interaction_id': 'count',  # counts duplicates
    'click_id': 'count'  # counts duplicates
}).rename(columns={'interaction_id': 'inaccurate_web_visits', 'click_id': 'inaccurate_ads_clicked'})

print("Inaccurate counts (with duplicates):")
print(inaccurate)

# Accurate counts (duplicates removed)
accurate = customers_visits_clicks_raw.groupby('customer_name').agg({
    'interaction_id': 'nunique',  # counts unique only
    'click_id': 'nunique'  # counts unique only
}).rename(columns={'interaction_id': 'web_visits', 'click_id': 'ads_clicked'})

print("\nAccurate counts (duplicates removed):")
print(accurate)

### Problem 4 (5 pts)

We will now work through a problem with outer joins.

Create a dataframe with the daily number of: 
* web visits
* unique web visitors
* unique customers visiting website
* ads clicks
* unique users clicking
* unique customers clicking

You should include the full date range 2024-05-01 to 2024-05-09 even though not all dates are included in either dataframe. 

You will create this data frame two ways:
1. Join `web_visits` to `ad_clicks` first and then perform the calculations
2. Create aggregate dataframes for `web_visits` to `ad_clicks` first and then join those aggregate dataframes together

#### Grading: 
5 points per method. 2 points for the correct join and 3 point for the aggragations. 

In [ ]:
# Method 1: Join first, then aggregate
method1_joined = web_visits.merge(ad_clicks, on='date', how='outer', suffixes=('_web', '_ad'))
method1_result = method1_joined.groupby('date').agg({
    'interaction_id': 'count',
    'visitor_id': 'nunique',
    'customer_id_web': lambda x: x.notna().sum(),
    'click_id': 'count',
    'external_user_id': 'nunique',
    'customer_id_ad': lambda x: x.notna().sum()
}).rename(columns={
    'interaction_id': 'web_visits',
    'visitor_id': 'unique_web_visitors',
    'customer_id_web': 'unique_customers_visiting',
    'click_id': 'ads_clicks',
    'external_user_id': 'unique_users_clicking',
    'customer_id_ad': 'unique_customers_clicking'
}).reset_index()

print("Method 1 - Join then aggregate:")
print(method1_result)

# Method 2: Aggregate first, then join
web_agg = web_visits.groupby('date').agg({
    'interaction_id': 'count',
    'visitor_id': 'nunique',
    'customer_id': lambda x: x.notna().sum()
}).rename(columns={
    'interaction_id': 'web_visits',
    'visitor_id': 'unique_web_visitors',
    'customer_id': 'unique_customers_visiting'
}).reset_index()

ad_agg = ad_clicks.groupby('date').agg({
    'click_id': 'count',
    'external_user_id': 'nunique',
    'customer_id': lambda x: x.notna().sum()
}).rename(columns={
    'click_id': 'ads_clicks',
    'external_user_id': 'unique_users_clicking',
    'customer_id': 'unique_customers_clicking'
}).reset_index()

method2_result = web_agg.merge(ad_agg, on='date', how='outer')

print("\nMethod 2 - Aggregate then join:")
print(method2_result)

# Create full date range
date_range = pd.date_range('2024-05-01', '2024-05-09')
method1_result['date'] = pd.to_datetime(method1_result['date'])
method2_result['date'] = pd.to_datetime(method2_result['date'])

method1_final = method1_result.set_index('date').reindex(date_range).reset_index().rename(columns={'index': 'date'}).fillna(0)
method2_final = method2_result.set_index('date').reindex(date_range).reset_index().rename(columns={'index': 'date'}).fillna(0)

print("\nMethod 1 - With full date range:")
print(method1_final)
print("\nMethod 2 - With full date range:")
print(method2_final)

### Problem 5 (15 pts)

Using the same two methods as problem 4 attempt to include:
* mean time on page per date 
* total time on page per date

One method will lead to some incorrect values unless you do a bit more work to correct the values. 

Write out a which method leads to correct values, which leads to incorrect values and why. 

#### Grading: 
* 5 points per method. 2 points for the correct join and 3 point for the aggragations. 
* 5 points for the written explaination. 

In [ ]:
# Method 1: Join first, then aggregate
method1_joined = web_visits.merge(ad_clicks, on='date', how='outer', suffixes=('_web', '_ad'))
method1_result_p5 = method1_joined.groupby('date').agg({
    'interaction_id': 'count',
    'visitor_id': 'nunique',
    'customer_id_web': lambda x: x.notna().sum(),
    'time_on_page': 'sum',
    'click_id': 'count',
    'external_user_id': 'nunique',
    'customer_id_ad': lambda x: x.notna().sum()
}).rename(columns={
    'interaction_id': 'web_visits',
    'visitor_id': 'unique_web_visitors',
    'customer_id_web': 'unique_customers_visiting',
    'time_on_page': 'total_time_on_page',
    'click_id': 'ads_clicks',
    'external_user_id': 'unique_users_clicking',
    'customer_id_ad': 'unique_customers_clicking'
}).reset_index()

# Calculate mean time on page (total / count of visits)
method1_result_p5['mean_time_on_page'] = method1_result_p5['total_time_on_page'] / method1_result_p5['web_visits']

print("Method 1 - Join then aggregate:")
print(method1_result_p5)

# Method 2: Aggregate first, then join
web_agg_p5 = web_visits.groupby('date').agg({
    'interaction_id': 'count',
    'visitor_id': 'nunique',
    'customer_id': lambda x: x.notna().sum(),
    'time_on_page': ['sum', 'mean']
}).reset_index()
web_agg_p5.columns = ['date', 'web_visits', 'unique_web_visitors', 'unique_customers_visiting', 'total_time_on_page', 'mean_time_on_page']

ad_agg_p5 = ad_clicks.groupby('date').agg({
    'click_id': 'count',
    'external_user_id': 'nunique',
    'customer_id': lambda x: x.notna().sum()
}).rename(columns={
    'click_id': 'ads_clicks',
    'external_user_id': 'unique_users_clicking',
    'customer_id': 'unique_customers_clicking'
}).reset_index()

method2_result_p5 = web_agg_p5.merge(ad_agg_p5, on='date', how='outer')

print("\nMethod 2 - Aggregate then join:")
print(method2_result_p5)

# Create full date range
date_range = pd.date_range('2024-05-01', '2024-05-09')
method1_result_p5['date'] = pd.to_datetime(method1_result_p5['date'])
method2_result_p5['date'] = pd.to_datetime(method2_result_p5['date'])

method1_final_p5 = method1_result_p5.set_index('date').reindex(date_range).reset_index().rename(columns={'index': 'date'}).fillna(0)
method2_final_p5 = method2_result_p5.set_index('date').reindex(date_range).reset_index().rename(columns={'index': 'date'}).fillna(0)

print("\nMethod 1 - With full date range:")
print(method1_final_p5)
print("\nMethod 2 - With full date range:")
print(method2_final_p5)

print("\n\nEXPLANATION:")
print("=" * 80)
print("Method 1 (Join then aggregate) leads to INCORRECT values.")
print("\nWhy: When we join web_visits and ad_clicks on date before aggregating,")
print("we create a Cartesian product for dates that have both web visits and ad clicks.")
print("For example, if a date has 3 web visits and 5 ad clicks, the join creates")
print("3 * 5 = 15 rows. When we sum 'time_on_page', we get 3 * (sum of 3 time values),")
print("which is 3x too large. Similarly, the mean calculation becomes incorrect.")
print("\nMethod 2 (Aggregate then join) leads to CORRECT values.")
print("Why: By aggregating web_visits and ad_clicks separately first, we preserve")
print("the correct counts and statistics for each source. Then joining the")
print("aggregated tables maintains the correct values since we're not creating")
print("a Cartesian product that inflates the time_on_page sums.")

### Problem 6 (15 pts)

Implement the full algorithm described in [Inverting Any Square Full-Rank Matrix](https://learning.oreilly.com/library/view/practical-linear-algebra/9781098120603/ch08.html#inverting-any-square) and reproduce Figure 8-3. Of course, your matrices will look different from Figure 8-3 because of random numbers, although the grid and identity matrices will be the same.

#### Grading: 
* 10 points for implimenting the algorithm. 
* 5 points for reproducing the figure. 

In [ ]:
# Problem 6: Implement the matrix inversion algorithm
# Following the algorithm from "Inverting Any Square Full-Rank Matrix"

import matplotlib.pyplot as plt

# Create a random square full-rank matrix
np.random.seed(42)
n = 4
A = np.random.randn(n, n)

# Make sure it's full-rank by adjusting if needed
while np.linalg.matrix_rank(A) < n:
    A = np.random.randn(n, n)

print("Original Matrix A:")
print(A)

# Algorithm: Create augmented matrix [A | I]
# Then perform row operations to transform A to I, which transforms I to A^-1

# Create identity matrix
I = np.eye(n)

# Create augmented matrix
augmented = np.hstack([A, I])
print("\nAugmented Matrix [A | I]:")
print(augmented)

# Forward elimination (Gauss-Jordan elimination)
for i in range(n):
    # Find pivot
    max_row = i
    for k in range(i + 1, n):
        if abs(augmented[k, i]) > abs(augmented[max_row, i]):
            max_row = k
    
    # Swap rows
    augmented[[i, max_row]] = augmented[[max_row, i]]
    
    # Scale pivot row
    augmented[i] = augmented[i] / augmented[i, i]
    
    # Eliminate column
    for k in range(n):
        if k != i:
            augmented[k] = augmented[k] - augmented[k, i] * augmented[i]

print("\nAugmented Matrix after Gauss-Jordan elimination [I | A^-1]:")
print(augmented)

# Extract the inverse matrix
A_inv_computed = augmented[:, n:]
print("\nComputed A^-1:")
print(A_inv_computed)

# Verify with numpy's inverse
A_inv_numpy = np.linalg.inv(A)
print("\nNumPy's A^-1:")
print(A_inv_numpy)

# Check: A * A^-1 should be approximately identity
print("\nVerification: A * A^-1 (should be close to Identity):")
print(A @ A_inv_computed)

# Visual representation like Figure 8-3
fig, axes = plt.subplots(1, 4, figsize=(15, 3))

axes[0].imshow(A, cmap='gray')
axes[0].set_title('Original Matrix A')
axes[0].set_xlabel('Column')
axes[0].set_ylabel('Row')

axes[1].imshow(I, cmap='gray')
axes[1].set_title('Identity Matrix I')
axes[1].set_xlabel('Column')
axes[1].set_ylabel('Row')

axes[2].imshow(A_inv_computed, cmap='gray')
axes[2].set_title('Computed A^-1')
axes[2].set_xlabel('Column')
axes[2].set_ylabel('Row')

axes[3].imshow(A @ A_inv_computed, cmap='gray')
axes[3].set_title('A * A^-1 (should be I)')
axes[3].set_xlabel('Column')
axes[3].set_ylabel('Row')

plt.tight_layout()
plt.savefig('matrix_inversion_visualization.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nFigure saved as 'matrix_inversion_visualization.png'")

### Problem 7 (15 pts)

The LIVE EVIL rule applies to the inverse of multiplied matrices. Test this in code by creating two square full-rank matrices $A$ and $B$, then use Euclidean distance to compare:
1. $(AB)^{-1}$ 
2. $A^{-1}B^{-1}$
3. $B^{-1}A^{-1}$

Before starting to code, make a prediction about which results will be equal. Print out your results using formatting like the following:

```
Distance between (AB)^-1 and (A^-1)(B^-1) is ___
Distance between (AB)^-1 and (B^-1)(A^-1) is ___
Distance between (A^-1)(B^-1) and (B^-1)(A^-1) is ___
```

#### Grading: 
5 points per comparision. 

In [ ]:
# Problem 7: Test the LIVE EVIL rule for matrix inverses
# LIVE EVIL rule: (AB)^-1 = B^-1 * A^-1 (note the reversed order, not A^-1 * B^-1)

import numpy as np
from scipy.spatial.distance import euclidean

# Prediction: 
# (AB)^-1 should equal B^-1 * A^-1 (distance close to 0)
# (AB)^-1 should NOT equal A^-1 * B^-1 (distance large)
# A^-1 * B^-1 should NOT equal B^-1 * A^-1 (distance large)

print("PREDICTION:")
print("(AB)^-1 and B^-1 * A^-1 should be equal (distance ≈ 0)")
print("(AB)^-1 and A^-1 * B^-1 should be different (distance >> 0)")
print("A^-1 * B^-1 and B^-1 * A^-1 should be different (distance >> 0)")
print("\n" + "="*70 + "\n")

# Create two random square full-rank matrices
np.random.seed(42)
size = 3
A = np.random.randn(size, size)
B = np.random.randn(size, size)

# Ensure they are full-rank
while np.linalg.matrix_rank(A) < size:
    A = np.random.randn(size, size)
while np.linalg.matrix_rank(B) < size:
    B = np.random.randn(size, size)

# Calculate the three expressions
AB_inv = np.linalg.inv(A @ B)
A_inv = np.linalg.inv(A)
B_inv = np.linalg.inv(B)

result_1 = AB_inv  # (AB)^-1
result_2 = A_inv @ B_inv  # A^-1 * B^-1
result_3 = B_inv @ A_inv  # B^-1 * A^-1

# Calculate Euclidean distances (flattened matrices)
distance_1 = euclidean(result_1.flatten(), result_3.flatten())
distance_2 = euclidean(result_1.flatten(), result_2.flatten())
distance_3 = euclidean(result_2.flatten(), result_3.flatten())

# Print results
print(f"Distance between (AB)^-1 and (A^-1)(B^-1) is {distance_2:.6f}")
print(f"Distance between (AB)^-1 and (B^-1)(A^-1) is {distance_1:.6f}")
print(f"Distance between (A^-1)(B^-1) and (B^-1)(A^-1) is {distance_3:.6f}")

print("\n" + "="*70)
print("CONCLUSION:")
print("="*70)
if distance_1 < 1e-10:
    print("✓ (AB)^-1 = (B^-1)(A^-1) - LIVE EVIL rule CONFIRMED")
else:
    print("✗ (AB)^-1 ≠ (B^-1)(A^-1)")
    
if distance_2 > 0.1:
    print("✓ (AB)^-1 ≠ (A^-1)(B^-1) - Order matters!")
else:
    print("✗ (AB)^-1 = (A^-1)(B^-1) - Unexpected!")
    
if distance_3 > 0.1:
    print("✓ (A^-1)(B^-1) ≠ (B^-1)(A^-1) - Matrix multiplication is not commutative")
else:
    print("✗ (A^-1)(B^-1) = (B^-1)(A^-1) - Unexpected!")